In [1]:
!pip install kafka-python

In [2]:
import json

from kafka import KafkaProducer

def json_serializer(data):
    return json.dumps(data).encode('utf-8')

server = 'localhost:9092'

producer = KafkaProducer(
    bootstrap_servers=[server],
    value_serializer=json_serializer
)

producer.bootstrap_connected()

True

In [1]:
import csv
import json
from time import time
from kafka import KafkaProducer

def main():

    # Create a Kafka producer
    producer = KafkaProducer(
        bootstrap_servers='localhost:9092',
        value_serializer=lambda v: json.dumps(v).encode('utf-8')
    )

    csv_file = 'data/green_tripdata_2019-10.csv'  # Update the path as needed

    # Define the columns to keep
    columns_to_keep = [
        'lpep_pickup_datetime',
        'lpep_dropoff_datetime',
        'PULocationID',
        'DOLocationID',
        'passenger_count',
        'trip_distance',
        'tip_amount'
    ]
    t0 = time()  # Start time
    with open(csv_file, 'r', newline='', encoding='utf-8') as file:
        reader = csv.DictReader(file)

        for row in reader:
            # Create a new dictionary with only the desired columns
            message = {col: row[col] for col in columns_to_keep if col in row}
            # Send data to Kafka topic "green-trips"
            producer.send('green-trips', value=message)

    # Flush any remaining messages and close the producer
    producer.flush()
    producer.close()

    t1 = time()  # End time
    took = t1 - t0
    print("Total time taken:", took, "seconds")

if __name__ == "__main__":
    main()


Total time taken: 51.750102043151855 seconds


In [5]:
df = pd.read_csv('data/green_tripdata_2019-10.csv')

/tmp/ipykernel_5974/3307007591.py:1: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('data/green_tripdata_2019-10.csv')


In [6]:
row_count = df.shape[0]
print("Number of rows:", row_count)

Number of rows: 476386


In [13]:
df = pd.read_csv('data/green_tripdata_2019-10.csv', nrows=100)

In [14]:
df

,VendorID,lpep_pickup_datetime,lpep_dropoff_datetime,store_and_fwd_flag,RatecodeID,PULocationID,DOLocationID,passenger_count,trip_distance,fare_amount,extra,mta_tax,tip_amount,tolls_amount,ehail_fee,improvement_surcharge,total_amount,payment_type,trip_type,congestion_surcharge
0,2,2019-10-01 00:26:02,2019-10-01 00:39:58,N,1,112,196,1,5.88,18.0,0.50,0.5,0.00,0.0,NaN,0.3,19.30,2,1,0.0
1,1,2019-10-01 00:18:11,2019-10-01 00:22:38,N,1,43,263,1,0.80,5.0,3.25,0.5,0.00,0.0,NaN,0.3,9.05,2,1,0.0
2,1,2019-10-01 00:09:31,2019-10-01 00:24:47,N,1,255,228,2,7.50,21.5,0.50,0.5,0.00,0.0,NaN,0.3,22.80,2,1,0.0
3,1,2019-10-01 00:37:40,2019-10-01 00:41:49,N,1,181,181,1,0.90,5.5,0.50,0.5,0.00,0.0,NaN,0.3,6.80,2,1,0.0
4,2,2019-10-01 00:08:13,2019-10-01 00:17:56,N,1,97,188,1,2.52,10.0,0.50,0.5,2.26,0.0,NaN,0.3,13.56,1,1,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,2,2019-10-01 00:02:53,2019-10-01 00:14:32,N,1,126,74,1,3.10,12.0,0.50,0.5,0.00,0.0,NaN,0.3,13.30,1,1,0.0
96,2,2019-10-01 00:18:45,2019-10-01 00:29:23,N,1,42,74,1,1.64,9.5,0.50,0.5,0.00,0.0,NaN,0.3,10.80,2,1,0.0
97,2,2019-10-01 00:41:32,2019-10-01 00:52:51,N,1,75,42,1,3.17,11.5,0.50,0.5,1.50,0.0,NaN,0.3,14.30,1,1,0.0
98,2,2019-10-01 00:36:54,2019-10-01 00:54:20,N,1,92,179,1,5.48,19.5,0.50,0.5,0.00,0.0,NaN,0.3,20.80,2,1,0.0


In [11]:
df.dtypes

VendorID                 float64
lpep_pickup_datetime      object
lpep_dropoff_datetime     object
store_and_fwd_flag        object
RatecodeID               float64
PULocationID               int64
DOLocationID               int64
passenger_count          float64
trip_distance            float64
fare_amount              float64
extra                    float64
mta_tax                  float64
tip_amount               float64
tolls_amount             float64
ehail_fee                float64
improvement_surcharge    float64
total_amount             float64
payment_type             float64
trip_type                float64
congestion_surcharge     float64
dtype: object

In [20]:
import pandas as pd
from kafka import KafkaProducer
import json
import time

def main():
    # Define the columns you want to keep
    columns_to_keep = [
        'lpep_pickup_datetime',
        'lpep_dropoff_datetime',
        'PULocationID',
        'DOLocationID',
        'passenger_count',
        'trip_distance',
        'tip_amount'
    ]

    # Load the CSV file while selecting the specified columns
    df = pd.read_csv('data/green_tripdata_2019-10.csv', usecols=columns_to_keep)

    # Convert the two datetime columns to timestamp
    df['lpep_pickup_datetime'] = pd.to_datetime(df['lpep_pickup_datetime'])
    df['lpep_dropoff_datetime'] = pd.to_datetime(df['lpep_dropoff_datetime'])

    # Initialize the KafkaProducer.
    # The value_serializer converts the message (dict) into a JSON formatted byte string.
    producer = KafkaProducer(
        bootstrap_servers=['localhost:9092'], 
        value_serializer=lambda v: json.dumps(v, default=str).encode('utf-8')
    )
    t0 = time.time()
    topic_name = 'green-trips' 

    # Iterate over the DataFrame rows and send each as a message to Kafka
    for _, row in df.iterrows():
        message = row.to_dict()
        producer.send(topic_name, message)
        # print("Sent message:", message)

    # Ensure all messages are sent before closing the producer
    producer.flush()
    t1 = time.time()
    print(f'Took {(t1 - t0):.2f} seconds')  # Print the total execution time
    producer.close()

if __name__ == '__main__':
    main()


Took 114.86 seconds


In [21]:
import pandas as pd
from kafka import KafkaProducer
import json
import time

def main():
    # Define the columns you want to keep
    columns_to_keep = [
        'lpep_pickup_datetime',
        'lpep_dropoff_datetime',
        'PULocationID',
        'DOLocationID',
        'passenger_count',
        'trip_distance',
        'tip_amount'
    ]

    # Load the CSV file while selecting the specified columns
    df = pd.read_csv('data/green_tripdata_2019-10.csv', usecols=columns_to_keep)

    # Convert the two datetime columns to proper datetime objects
    df['lpep_pickup_datetime'] = pd.to_datetime(df['lpep_pickup_datetime'])
    df['lpep_dropoff_datetime'] = pd.to_datetime(df['lpep_dropoff_datetime'])
    
    # Filter out rows that contain NaN values in the specified columns
    df = df.dropna(subset=columns_to_keep)
    
    # Initialize the KafkaProducer.
    # The value_serializer converts the message (dict) into a JSON formatted byte string.
    producer = KafkaProducer(
        bootstrap_servers=['localhost:9092'], 
        value_serializer=lambda v: json.dumps(v, default=str).encode('utf-8')
    )
    
    t0 = time.time()
    topic_name = 'green-trips' 

    # Iterate over the DataFrame rows and send each as a message to Kafka
    for _, row in df.iterrows():
        message = row.to_dict()
        producer.send(topic_name, message)
        # print("Sent message:", message)

    # Ensure all messages are sent before closing the producer
    producer.flush()
    t1 = time.time()
    print(f'Took {(t1 - t0):.2f} seconds')  # Print the total execution time
    producer.close()

if __name__ == '__main__':
    main()


Took 90.31 seconds
